# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore).
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv('../../datasets/day-of-week-not-scaled.csv')
#print(df.columns)
df_col = pd.read_csv('../../datasets/dayofweek.csv')
#print(df_col.columns)

df['dayofweek'] = df_col['dayofweek']
df.info()
#df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686 entries, 0 to 1685
Data columns (total 44 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   numTrials         1686 non-null   int64  
 1   hour              1686 non-null   int64  
 2   uid_user_0        1686 non-null   float64
 3   uid_user_1        1686 non-null   float64
 4   uid_user_10       1686 non-null   float64
 5   uid_user_11       1686 non-null   float64
 6   uid_user_12       1686 non-null   float64
 7   uid_user_13       1686 non-null   float64
 8   uid_user_14       1686 non-null   float64
 9   uid_user_15       1686 non-null   float64
 10  uid_user_16       1686 non-null   float64
 11  uid_user_17       1686 non-null   float64
 12  uid_user_18       1686 non-null   float64
 13  uid_user_19       1686 non-null   float64
 14  uid_user_2        1686 non-null   float64
 15  uid_user_20       1686 non-null   float64
 16  uid_user_21       1686 non-null   float64


In [4]:
'''from sklearn.preprocessing import StandardScaler
continuous_columns = ['numTrials', 'hour']
scaler = StandardScaler()
continuous_data_scaled = scaler.fit_transform(df[continuous_columns])
scaled_df = pd.DataFrame(continuous_data_scaled, columns=continuous_columns)
scaled_df = df.drop(columns=continuous_columns).join(scaled_df)
df = scaled_df'''

"from sklearn.preprocessing import StandardScaler\ncontinuous_columns = ['numTrials', 'hour']\nscaler = StandardScaler()\ncontinuous_data_scaled = scaler.fit_transform(df[continuous_columns])\nscaled_df = pd.DataFrame(continuous_data_scaled, columns=continuous_columns)\nscaled_df = df.drop(columns=continuous_columns).join(scaled_df)\ndf = scaled_df"

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}



In [7]:
from sklearn.svm import SVC
model = SVC(random_state=21, probability=True)

In [8]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs = -1)
grid_search.fit(X_train, y_train)

/home/morrowto/.local/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)
/home/morrowto/.local/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)
/home/morrowto/.local/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indic

GridSearchCV(estimator=SVC(probability=True, random_state=21), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']})

In [9]:
results = pd.DataFrame(grid_search.cv_results_)
results_sorted = results.sort_values(by='rank_test_score')

In [10]:
print(results_sorted[['params', 'rank_test_score']].head())
best_model = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

                                               params  rank_test_score
70  {'C': 10, 'class_weight': None, 'gamma': 'auto...                1
64  {'C': 10, 'class_weight': 'balanced', 'gamma':...                2
58  {'C': 5, 'class_weight': None, 'gamma': 'auto'...                3
52  {'C': 5, 'class_weight': 'balanced', 'gamma': ...                4
60  {'C': 10, 'class_weight': 'balanced', 'gamma':...                5

Best parameters: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best cross-validation accuracy: 0.8761


In [11]:
from sklearn.metrics import accuracy_score
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {test_accuracy:.4f}")

top_scores = results_sorted['mean_test_score'].head(5)
score_diff = top_scores.iloc[0] - top_scores.iloc[4]
print(f"\nDifference between top and 5th best model: {score_diff:.4f}")
if score_diff < 0.02:
    print("Consider simpler models with comparable performance")

Test set accuracy: 0.8876

Difference between top and 5th best model: 0.1551


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [12]:
param_grid = {
    'max_depth' : np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion' : ['entropy', 'gini']
}

In [13]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=21)

In [14]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs = -1)
grid_search.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])})

In [15]:
results = pd.DataFrame(grid_search.cv_results_)
results_sorted = results.sort_values(by='rank_test_score')

In [16]:
print(results_sorted[['params', 'rank_test_score']].head())
best_model = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

                                               params  rank_test_score
70  {'class_weight': 'balanced', 'criterion': 'gin...                1
69  {'class_weight': 'balanced', 'criterion': 'gin...                2
80  {'class_weight': 'balanced', 'criterion': 'gin...                3
81  {'class_weight': 'balanced', 'criterion': 'gin...                3
96  {'class_weight': 'balanced', 'criterion': 'gin...                3

Best parameters: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': np.int64(22)}
Best cross-validation accuracy: 0.8731


In [17]:
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {test_accuracy:.4f}")

top_scores = results_sorted['mean_test_score'].head(5)
score_diff = top_scores.iloc[0] - top_scores.iloc[4]
print(f"\nDifference between top and 5th best model: {score_diff:.4f}")
if score_diff < 0.02:
    print("Consider simpler models with comparable performance")

Test set accuracy: 0.8905

Difference between top and 5th best model: 0.0000
Consider simpler models with comparable performance


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [18]:
param_grid = {
    'n_estimators':[5, 10, 50, 100],
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion' : ['entropy', 'gini']
}

In [19]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=21)

In [20]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs = -1)
grid_search.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]),
                         'n_estimators': [5, 10, 50, 100]})

In [21]:
print(results_sorted[['params', 'rank_test_score']].head())
best_model = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

                                               params  rank_test_score
70  {'class_weight': 'balanced', 'criterion': 'gin...                1
69  {'class_weight': 'balanced', 'criterion': 'gin...                2
80  {'class_weight': 'balanced', 'criterion': 'gin...                3
81  {'class_weight': 'balanced', 'criterion': 'gin...                3
96  {'class_weight': 'balanced', 'criterion': 'gin...                3

Best parameters: {'class_weight': None, 'criterion': 'gini', 'max_depth': np.int64(28), 'n_estimators': 50}
Best cross-validation accuracy: 0.9043


In [22]:
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {test_accuracy:.4f}")

top_scores = results_sorted['mean_test_score'].head(5)
score_diff = top_scores.iloc[0] - top_scores.iloc[4]
print(f"\nDifference between top and 5th best model: {score_diff:.4f}")
if score_diff < 0.02:
    print("Consider simpler models with comparable performance")

Test set accuracy: 0.9290

Difference between top and 5th best model: 0.0000
Consider simpler models with comparable performance


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [24]:
from sklearn.model_selection import cross_val_score
from tqdm.notebook import tqdm
from itertools import product

In [26]:
results = []

# tqdm.notebook показывает красивый интерактивный прогресс-бар
progress_bar = tqdm(total=len(param_grid['max_depth'])*len(param_grid['class_weight'])*len(param_grid['criterion']),
                   desc="Выполнение GridSearch")

for max_depth in param_grid['max_depth']:
    for class_weight in param_grid['class_weight']:
        for criterion in param_grid['criterion']:
            model = RandomForestClassifier(n_estimators=100, max_depth=max_depth, class_weight=class_weight, 
                                           criterion=criterion, random_state=21,n_jobs=-1)
            scores = cross_val_score(estimator=model, X=X_train, y=y_train, cv=5, scoring='accuracy', n_jobs=-1)
            results.append({'max_depth': max_depth, 'class_weight': class_weight, 'criterion': criterion,
                'mean_accuracy': np.mean(scores), 'std_accuracy': np.std(scores)})
            
            progress_bar.update(1)

progress_bar.close()

results_df = pd.DataFrame(results)
sorted_results = results_df.sort_values('mean_accuracy', ascending=False)
print("Топ-5 лучших комбинаций параметров:")
print(sorted_results.head())
top_accuracy = sorted_results['mean_accuracy'].iloc[0]
fifth_accuracy = sorted_results['mean_accuracy'].iloc[4]
accuracy_diff = top_accuracy - fifth_accuracy

print(f"\nРазница между лучшей и 5-й моделью: {accuracy_diff:.4f}")

if accuracy_diff < 0.02:
    print("\nРекомендация: несколько моделей работают почти одинаково хорошо.")
    print("Можно выбрать более простую модель из топ-5.")
    
    simplest_model = sorted_results.head(5).sort_values('max_depth').iloc[0]
    print("\nСамая простая модель в топ-5:")
    print(simplest_model)

Выполнение GridSearch:   0%|          | 0/196 [00:00<?, ?it/s]

Топ-5 лучших комбинаций параметров:
     max_depth class_weight criterion  mean_accuracy  std_accuracy
123         31         None      gini       0.903547       0.01438
175         44         None      gini       0.902806       0.01046
171         43         None      gini       0.902806       0.01046
179         45         None      gini       0.902806       0.01046
183         46         None      gini       0.902806       0.01046

Разница между лучшей и 5-й моделью: 0.0007

Рекомендация: несколько моделей работают почти одинаково хорошо.
Можно выбрать более простую модель из топ-5.

Самая простая модель в топ-5:
max_depth              31
class_weight         None
criterion            gini
mean_accuracy    0.903547
std_accuracy      0.01438
Name: 123, dtype: object


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [27]:
best_params = sorted_results.iloc[0]
best_params

max_depth              31
class_weight         None
criterion            gini
mean_accuracy    0.903547
std_accuracy      0.01438
Name: 123, dtype: object

In [28]:
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=best_params['max_depth'],
    class_weight=best_params['class_weight'],
    criterion=best_params['criterion'],
    random_state=21,
    n_jobs=-1
)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)


In [30]:
final_accuracy = accuracy_score(y_test, y_pred)
final_accuracy

0.9378698224852071